# Market Regime Detection via Data Smashing 2.0

This notebook uses **lsmash** to find **cross-temporal relationships** in financial market data.

The core idea: encode each year's daily return sequence as a stream of discrete symbols
(bear / neutral / bull days), then compute pairwise Sequence Likelihood divergences.
Years with similar *statistical structure* — not just similar returns — will cluster
together in the distance matrix.

**What to expect:**
- Crisis years (2001–02, 2008–09, 2020, 2022) should form their own cluster
- Quiet bull-market years should cluster separately
- The dendrogram reveals market *regime similarity* that simple correlation misses

**Part 1** — Annual regime comparison: S&P 500, 2000–2023  
**Part 2** — Cross-asset structural comparison: which asset classes share a common regime signature?

## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import yfinance as yf
import lsmash as ls
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

# Years shaded as known crisis / stress periods
CRISIS_YEARS = {2001, 2002, 2008, 2009, 2020, 2022}

def make_lsmash_opts():
    opt = ls.LsmashOptions()
    opt.data_type  = 'symbolic'
    opt.data_dir   = 'row'
    opt.sae        = True
    opt.num_repeat = 30
    opt.depth      = 8
    return opt

---
## Part 1 — Annual Regime Comparison: S&P 500 (2000–2023)

### 1a. Download and discretise

In [ ]:
print('Downloading S&P 500 daily data...')
sp500 = yf.download('^GSPC', start='2000-01-01', end='2024-01-01', progress=False)
closes = sp500['Close'].squeeze().dropna()

# Daily log returns
log_ret = np.log(closes / closes.shift(1)).dropna()
print(f'  {len(log_ret):,} trading days from {log_ret.index[0].date()} to {log_ret.index[-1].date()}')

# Discretise into 3 symbols using global tertile thresholds:
#   0 = bear day (bottom third of all returns)
#   1 = neutral  (middle third)
#   2 = bull day (top third)
q33, q67 = np.percentile(log_ret, [33.3, 66.7])
print(f'  Thresholds: bear < {q33:.4f} < neutral < {q67:.4f} < bull')

def discretise(series):
    """Returns list of unsigned ints {0, 1, 2}."""
    return np.digitize(series.values, bins=[q33, q67]).tolist()

# Split by calendar year — require at least 200 trading days
annual = {}
for year in range(2000, 2024):
    yr = log_ret[log_ret.index.year == year]
    if len(yr) >= 200:
        annual[year] = discretise(yr)

labels_yr = [str(y) for y in annual.keys()]
seqs_yr   = list(annual.values())
print(f'\n{len(seqs_yr)} annual sequences ({labels_yr[0]}–{labels_yr[-1]})')
print(f'Sequence lengths: {min(len(s) for s in seqs_yr)}–{max(len(s) for s in seqs_yr)} trading days')

### 1b. Compute pairwise SL divergence

In [ ]:
print('Running lsmash on annual S&P 500 sequences...')
D_yr = ls.from_sequences(seqs_yr, make_lsmash_opts())
print(f'Distance matrix shape: {D_yr.shape}')

### 1c. Visualise — heatmap and dendrogram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# ── Heatmap ──────────────────────────────────────────────────────────────
ax = axes[0]
sns.heatmap(
    D_yr, xticklabels=labels_yr, yticklabels=labels_yr,
    cmap='magma_r', ax=ax, linewidths=0.3, linecolor='#333',
    cbar_kws={'label': 'SL Divergence'},
)
ax.set_title('S&P 500 — Annual Regime Distance Matrix', fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=9)
# Highlight crisis years in red
for tick in ax.get_xticklabels() + ax.get_yticklabels():
    if int(tick.get_text()) in CRISIS_YEARS:
        tick.set_color('crimson')
        tick.set_fontweight('bold')

# ── Dendrogram ────────────────────────────────────────────────────────────
ax2 = axes[1]
condensed = squareform(D_yr, checks=False)
Z = linkage(condensed, method='average')
dend = dendrogram(Z, labels=labels_yr, ax=ax2, leaf_font_size=10,
                  leaf_rotation=45, color_threshold=0, above_threshold_color='steelblue')
ax2.set_title('Hierarchical Clustering of Market Regimes (UPGMA)', fontsize=12, fontweight='bold')
ax2.set_ylabel('SL Divergence')
# Colour crisis year labels red
for tick in ax2.get_xticklabels():
    if int(tick.get_text()) in CRISIS_YEARS:
        tick.set_color('crimson')
        tick.set_fontweight('bold')

legend = [
    mpatches.Patch(color='crimson',   label='Crisis / stress year'),
    mpatches.Patch(color='steelblue', label='Other year'),
]
axes[1].legend(handles=legend, loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('market_regimes_annual.png', dpi=150, bbox_inches='tight')
plt.show()

### 1d. Most similar years (nearest neighbours)

In [ ]:
df_yr = pd.DataFrame(D_yr, index=labels_yr, columns=labels_yr)

print('Top-3 most similar years to each crisis year:\n')
for cy in [y for y in labels_yr if int(y) in CRISIS_YEARS]:
    neighbors = df_yr[cy].drop(cy).nsmallest(3)
    neighbor_str = ', '.join(f"{yr} ({d:.4f})" for yr, d in neighbors.items())
    print(f'  {cy}: {neighbor_str}')

---
## Part 2 — Cross-Asset Structural Comparison

Which asset classes share the most similar regime signature over the same period?
We compare five major instruments — equities, gold, crude oil, the VIX fear gauge,
and long-term Treasury yields — each encoded as a full-history symbol stream.

### 2a. Download and discretise all assets

In [ ]:
ASSETS = {
    'S&P 500':   '^GSPC',
    'Gold':      'GC=F',
    'Crude Oil': 'CL=F',
    'VIX':       '^VIX',
    'US 10Y':    '^TNX',
}

asset_seqs, asset_labels = [], []
print('Downloading asset data (2005–2023)...')

for name, ticker in ASSETS.items():
    raw = yf.download(ticker, start='2005-01-01', end='2024-01-01',
                      progress=False, auto_adjust=True)
    prices = raw['Close'].squeeze().dropna()
    if len(prices) < 500:
        print(f'  {name}: insufficient data, skipping')
        continue
    ret = np.log(prices / prices.shift(1)).dropna()
    # Per-asset tertile thresholds (each asset has its own volatility scale)
    q33a, q67a = np.percentile(ret, [33.3, 66.7])
    sym = np.digitize(ret.values, bins=[q33a, q67a]).tolist()
    asset_seqs.append(sym)
    asset_labels.append(name)
    print(f'  {name} ({ticker}): {len(sym):,} days')

print(f'\n{len(asset_seqs)} assets ready.')

### 2b. Compute pairwise distances

In [ ]:
print('Running lsmash on cross-asset sequences...')
D_assets = ls.from_sequences(asset_seqs, make_lsmash_opts())
print(f'Distance matrix shape: {D_assets.shape}')

df_assets = pd.DataFrame(D_assets, index=asset_labels, columns=asset_labels)
print('\nPairwise SL Divergence Matrix:')
print(df_assets.round(4).to_string())

### 2c. Visualise

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(
    D_assets, annot=True, fmt='.4f', cmap='viridis',
    xticklabels=asset_labels, yticklabels=asset_labels,
    ax=axes[0], linewidths=0.5,
    cbar_kws={'label': 'SL Divergence'},
)
axes[0].set_title('Cross-Asset Regime Distance (2005–2023)', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)

# Dendrogram
condensed_a = squareform(D_assets, checks=False)
Za = linkage(condensed_a, method='average')
dendrogram(Za, labels=asset_labels, ax=axes[1], leaf_font_size=11,
           color_threshold=0, above_threshold_color='darkorange')
axes[1].set_title('Cross-Asset Hierarchical Clustering', fontsize=12, fontweight='bold')
axes[1].set_ylabel('SL Divergence')

plt.tight_layout()
plt.savefig('market_regimes_assets.png', dpi=150, bbox_inches='tight')
plt.show()